## 1. 환경 설정 및 라이브러리 임포트

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

# API 키 확인
if openai_api_key:
    print(f"✅ OpenAI API 키 로드 완료: sk-...{openai_api_key[-4:]}")
else:
    print("❌ API 키가 없습니다. .env 파일을 확인하세요.")

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ OpenAI API 키 로드 완료: sk-...yK4A


## 2. Document Upload

In [2]:
file_paths = [
    "C:/Users/user/Downloads/삼성_2025Q4_conference_eng_presentation.pdf",
    "C:/Users/user/Downloads/삼성_2025Q4_script_eng_AudioScript.pdf"
]

docs = []

for path in file_paths:
    # 각 파일을 로드
    loader = PyPDFLoader(path)
    
    # load를 통해 문서의 각 페이지를 Document 객체로 변환
    docs.extend(loader.load())

## 3. Text Splitting

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    
    chunk_overlap=220
)

splits = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(splits)}\n")
print(f"첫번째 청크 예시:\n{splits[0].page_content}\n")
print(f"두번째 청크 예시:\n{splits[1].page_content}")

분할된 청크 수: 78

첫번째 청크 예시:
SAMSUNG 
ELECTRONICS
Earnings Presentation: 
4Q 2025 Financial Results

두번째 청크 예시:
The financial information in this document are consolidated earnings results based on K-IFRS.
This document is provided for the convenience of investors only before the external audit on our 4Q 2025 financial results iscompleted. 
The Audit outcomes may cause some parts of this document to change. 
This document contains "forward-looking statements" - that is statements related to future not past events. 
In this context "forward-looking statements" often address our expected future business and financial performance 
and often contain words such as "expects" "anticipates" "intends" "plans" "believes" "seeks" or "will". 
"Forward-looking statements" by their nature address matters that are to different degrees uncertain. 
For us particular uncertainties which could adversely or positively affect our future results include:
·  The behavior of financial markets including fluctuatio

## 4. Embedding


In [4]:
# OpenAI 임베딩 (API 호출 방식 — GPU 불필요)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Chroma 벡터스토어
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

## 5. LLM 연결


In [5]:
chat_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=openai_api_key
)

## 6. Query-Translation: Multi-Query


In [6]:
# RAG-Fusion: Related
template = """
You are a helpful assistant that generates multiple search queries based on a single input query.
The target documents are written in English.
Even if the input query is in Korean, please generate 4 search queries in English to improve retrieval performance.

Input Query: {question}
Output (4 English queries):
"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [7]:
generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)
question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘"

# 쿼리 생성 확인
queries = generate_queries.invoke({"question": question})
print("=== 생성된 쿼리들 ===")
for q in queries:
    print(q, "\n")

=== 생성된 쿼리들 ===
1. Samsung Electronics 2025 Q4 financial results announcement 

2. Samsung Electronics fourth quarter 2025 performance report 

3. Samsung Electronics Q4 2025 earnings release 

4. Information on Samsung Electronics 2025 fourth quarter results announcement 



## 7. Query-Translation: RAG-Fusion


In [8]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}
    doc_map ={}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            content_key = doc.page_content.strip()
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if content_key not in fused_scores:
                fused_scores[content_key] = 0
                doc_map[content_key] = doc
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[content_key] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (doc_map[key], score)
        for key, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

6

## 8. RAG Chain

In [9]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant specializing in Samsung Electronics earnings reports.
Answer the question based ONLY on the provided context.
If the context does not contain enough information to answer confidently,
say "The provided context does not contain sufficient information on this topic."
Always include specific numbers and financial figures when available.

#Context:
{context}

#Question:
{question}

#Answer:
""")

def format_docs_from_fusion(results):
    """RRF 결과 (Document, score) 튜플에서 텍스트만 추출"""
    return "\n\n".join(doc.page_content for doc, score in results)

final_rag_chain = (
    {
        "context": itemgetter("question") | retrieval_chain_rag_fusion | format_docs_from_fusion,
        "question": itemgetter("question")
    }
    | RAG_PROMPT
    | chat_llm
    | StrOutputParser()
)

## 9. 실행 및 테스트

In [ ]:
from langchain_teddynote.messages import stream_response

question = "삼성전자 2025 4분기 실적 발표에 대해서 알려줘."
answer = final_rag_chain.stream({"question": question})
stream_response(answer)
# Langsmith Trace
# URL: https://smith.langchain.com/public/cd8890dc-7fb5-48d3-975f-15aa305b9b50/r

삼성전자의 2025년 4분기 실적 발표에 따르면, 회사는 역대 최고 분기 매출인 93.8조 원을 기록하였으며, 이는 전 분기 대비 9% 증가한 수치입니다. 운영 이익은 20.1조 원으로, 전 분기 대비 7.3% 포인트 증가하여 운영 마진은 21.4%에 도달했습니다. 

DX 부문에서는 매출이 8% 감소했으나, DS 부문은 33% 증가하여 메모리 제품의 판매 증가로 인해 강한 성과를 보였습니다. SG&A 비용은 24.2조 원으로, 전 분기 대비 2.9조 원 증가하였고, R&D 투자도 10.9조 원으로 증가했습니다. 

2025년 전체 실적은 매출 333.6조 원, 운영 이익 43.6조 원으로 집계되었습니다.